# IPC2BNS-Verify — Phase 2: Ingestion & Retrieval Layer

This notebook verifies Phase 2 deliverables:
1. **Section-Level Chunker** (`chunker.py`): Structured chunks with temporal validity metadata.
2. **Cleaned Corpora**: `ipc_sections.jsonl` (145 provisions) and `bns_sections.jsonl` (130 provisions).
3. **Statutory Vector Index** (`embedder.py`): BM25/hybrid similarity engine.
4. **Retrieval Search Engine** (`search.py`): Top-k retrieval with temporal validity filtering.
5. **Benchmark Dataset Evaluation** (`retrieval_eval.py`): Computes Recall@k, Precision@k, and MRR.
6. **Automated Pytest Suite**: Full test run.

---
## 1. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment initialized.')

---
## 2. Dependencies

In [ ]:
!pip install -q pytest
print('Pytest ready.')

---
## 3. Inspect Cleaned Statutory Corpora & Chunks

In [ ]:
from src.ingestion.chunker import load_all_chunks

cleaned_dir = os.path.join(PROJECT_ROOT, 'data/01_cleaned')
chunks_dict = load_all_chunks(cleaned_dir)

print(f'IPC Chunks Loaded: {len(chunks_dict["IPC"])}')
print(f'BNS Chunks Loaded: {len(chunks_dict["BNS"])}')
print(f'Total Corpus Size: {len(chunks_dict["ALL"])} sections\n')

# Sample chunk
sample = chunks_dict['BNS'][0]
print('--- Sample Statutory Chunk ---')
print(f'ID      : {sample.chunk_id}')
print(f'Act     : {sample.act_full_name}')
print(f'Section : §{sample.section_number} - {sample.section_title}')
print(f'Dates   : {sample.effective_start} to {sample.effective_end}')
print(f'Text    : {sample.section_text[:120]}...')

---
## 4. Build / Load Statutory Vector Index

In [ ]:
from src.retrieval.embedder import build_and_save_index, LocalStatutoryVectorIndex

index_dir = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage2_index')
index = build_and_save_index(cleaned_dir, index_dir)
print('Vector index successfully built and persisted.')

---
## 5. Interactive Statutory Retrieval Queries

In [ ]:
from src.retrieval.search import retrieve_statutes

queries = [
    ('What is the punishment for murder under BNS?', 'BNS'),
    ('Where is snatching or petty theft penalized?', 'BNS'),
    ('Penalty for rash driving causing death (hit and run)?', 'BNS'),
    ('What section defines cheating in IPC?', 'IPC'),
    ('Provisions for terrorist acts under new law?', 'BNS'),
]

for q, act in queries:
    print('='*75)
    print(f'Query: "{q}" [Filter: {act}]')
    print('='*75)
    hits = retrieve_statutes(q, top_k=2, act_filter=act)
    for rank, h in enumerate(hits, start=1):
        print(f'  [{rank}] {h["act"]} §{h["section_number"]}: {h["section_title"]} (score: {h["similarity_score"]:.2f})')
        print(f'      Text: {h["section_text"][:100]}...')
    print()

---
## 6. Temporal Validity Filtering Demonstration

Demonstrates the **TaxFlow-inspired temporal validity filtering**: queries set before July 1, 2024 retrieve IPC, while queries set after retrieve BNS.

In [ ]:
from src.retrieval.search import get_retriever
retriever = get_retriever()

print('--- Conduct Date: 2021-05-15 (Pre-Transition -> IPC Applies) ---')
hits_2021 = retriever.retrieve('murder', top_k=2, target_date='2021-05-15')
for h in hits_2021:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

print('\n--- Conduct Date: 2025-01-10 (Post-Transition -> BNS Applies) ---')
hits_2025 = retriever.retrieve('murder', top_k=2, target_date='2025-01-10')
for h in hits_2025:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

---
## 7. Evaluate Retrieval Accuracy (Precision, Recall, MRR)

In [ ]:
from src.eval.retrieval_eval import evaluate_retrieval

benchmark_dev = os.path.join(PROJECT_ROOT, 'data/03_benchmark/benchmark_dev.csv')
metrics_out = os.path.join(PROJECT_ROOT, 'results/stage2/retrieval_metrics.json')

metrics = evaluate_retrieval(benchmark_dev, metrics_out, top_k=5)

print('\n' + '='*50)
print('STAGE 2 RETRIEVAL METRICS SUMMARY')
print('='*50)
print(json.dumps(metrics['metrics'], indent=2))

---
## 8. Run Full Automated Test Suite

In [ ]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

---
## 9. Check WBS Completion

In [ ]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report